In [1]:
%cd /home/qid/MMMM

/home/qid/MMMM


In [2]:
import sys
sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")
sys.path.append("./trainer")
import time

In [3]:
# Import Pytorch
import torch
import torch.nn as nn

import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

# Import Pretrain Libraries (transformers + diffusers)
from transformers import BartTokenizer
from diffusers import AutoencoderKL

# Parallel Helper
# from parallel import DataParallelModel, DataParallelCriterion

# Import Our Own Functions
from master_init import *
from DSG import *

from count_params import count_params

In [4]:
device="cuda"
device_ids=[0,1,2,3]
model = INITIALIZE_MODEL(device=None, device_ids=device_ids, dtype=torch.float32)
model = nn.DataParallel(model, device_ids).to(device)

In [5]:
dataset_dict = INITIALIZE_DATALOADERS(
        keys=["Brain2Image"],
        bsz=[4],
        dev_bsz=[4]
    )

[WARNING] Batch Size for Brain2Image AKA ImageNet is NOT 1. UNEXPECTED BEHAVIOR MAY OCCUR.


In [6]:
dataloader = dataset_dict["Brain2Image"]["train"]

In [7]:
current_data = dataloader.load_data()

In [8]:
max_len = max(i.shape[0] for i in current_data["data"])
eegs = []
masks = []
invert_masks = []
for i in range(len(current_data["data"])):
    eeg = current_data["data"][i]
    cur_sz = eeg.shape[0]
    mask = torch.cat((torch.ones(cur_sz), torch.zeros(max_len - cur_sz)))
    masks.append(mask)
    invert_masks.append(1 - mask)
    eegs.append(torch.cat((eeg, torch.zeros(max_len - cur_sz, eeg.shape[1]))))

eeg_batch = torch.stack(eegs)
masks_batch = torch.stack(masks)
invert_masks_batch = torch.stack(invert_masks)

In [9]:
labels = current_data["labels"]
latents = current_data["latents"]
latents = torch.cat(latents, dim=0)

In [10]:
noise = torch.randn_like(latents)
bsz = latents.shape[0]

In [11]:
timesteps = torch.randint(0, 30, (bsz,))
timesteps = timesteps.long()

noisy_latents = latents + noise

In [13]:
noisy_latents.shape

torch.Size([4, 4, 64, 64])

In [12]:
args_dict = {
    "input_data_batch" : eeg_batch,
    "input_masks_batch" : masks_batch,
    "input_masks_invert" : invert_masks_batch,
    "pool_result" : True,
    "noisy_latents" : noisy_latents.to(dtype=torch.float32),
    "timesteps" : timesteps,
    "train" : True
}
model_pred = model("EEG-IMG-DIFFUSION", args_dict)

In [15]:
model_pred.shape

torch.Size([4, 4, 64, 64])

In [20]:
nn.MSELoss(reduction="none")(noise, model_pred, dim=0)

TypeError: MSELoss.forward() got an unexpected keyword argument 'dim'